In [21]:
import os
from strands.models.gemini import GeminiModel
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")

model = GeminiModel(
    client_args={
        "vertexai": True,
    },
    model_id=MODEL_ID,
)

print(f"Using {MODEL_ID} on Gemini")

Using gemini-3.8-flash on Gemini


In [11]:
# os.environ["TAVILY_API_KEY"] = "tvly-..."
# os.environ["ALPHA_VANTAGE_API_KEY"] = "..."

if not os.getenv("TAVILY_API_KEY"):
    print("⚠️  TAVILY_API_KEY not set — the news and sentiment agents cannot search the web.")
else:
    print("✓ TAVILY_API_KEY is configured")

if not os.getenv("ALPHA_VANTAGE_API_KEY"):
    print("ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).")
else:
    print("✓ ALPHA_VANTAGE_API_KEY is configured")

✓ TAVILY_API_KEY is configured
ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).


In [13]:
import yfinance as yf
from strands import Agent, ToolContext, tool
from strands_tools.tavily import tavily_search


# Groq's free tier rejects any single request over 8,000 tokens ("Request too large").
# A default Tavily search returns ~3,700 tokens, so a worker that searches twice overflows.
# The workers use this wrapper instead: the same Tavily search, top 3 results (~1,100 tokens),
# at most twice per task, because every result is resent with the worker's next request.
@tool(context=True)
async def web_search(query: str, tool_context: ToolContext, time_range: str = "month") -> dict | str:
    """Search the web with Tavily and return the top 3 results.

    Args:
        query: What to search for
        time_range: How recent the results should be: day, week, month or year
    """
    searches = tool_context.agent.state.get("searches") or 0  # as_tool() resets state for each task
    if searches >= 2:
        return "Search limit reached: answer with the results you already have."
    tool_context.agent.state.set("searches", searches + 1)
    return await tavily_search(query=query, max_results=3, time_range=time_range)

In [15]:
def _num(value, money=False, pct=False):
    """Format numbers safely — Yahoo sometimes leaves fields empty."""
    if not isinstance(value, (int, float)):
        return "N/A"
    if pct:
        return f"{value * 100:.2f}%"
    return f"${value:,.2f}" if money else f"{value:,.2f}"


@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price and key metrics using Yahoo Finance.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA', 'MSFT')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Stock: {symbol}\n"
            f"Current Price: {_num(info.get('currentPrice') or info.get('regularMarketPrice'), money=True)}\n"
            f"Market Cap: {_num(info.get('marketCap'), money=True)}\n"
            f"P/E Ratio: {_num(info.get('trailingPE'))}\n"
            f"52 Week High: {_num(info.get('fiftyTwoWeekHigh'), money=True)}\n"
            f"52 Week Low: {_num(info.get('fiftyTwoWeekLow'), money=True)}\n"
            f"Average Volume: {_num(info.get('averageVolume'))}\n"
            f"Sector: {info.get('sector', 'N/A')}\n"
            f"Industry: {info.get('industry', 'N/A')}"
        )
    except Exception as e:
        return f"Error fetching stock data for {symbol}: {e}"


@tool
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """Get historical stock price data.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
        period: One of 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    """
    try:
        hist = yf.Ticker(symbol).history(period=period)
        if hist.empty:
            return f"No historical data available for {symbol}"
        start_price = hist["Close"].iloc[0]
        end_price = hist["Close"].iloc[-1]
        change_pct = (end_price - start_price) / start_price * 100
        return (
            f"Historical Data for {symbol} ({period}):\n"
            f"Starting Price: ${start_price:.2f}\n"
            f"Ending Price: ${end_price:.2f}\n"
            f"Change: {change_pct:+.2f}%\n"
            f"Highest: ${hist['High'].max():.2f}\n"
            f"Lowest: ${hist['Low'].min():.2f}\n"
            f"Average Volume: {hist['Volume'].mean():,.0f}\n\n"
            f"Recent 5-day trend:\n{hist[['Close', 'Volume']].tail().to_string()}"
        )
    except Exception as e:
        return f"Error fetching historical data for {symbol}: {e}"


@tool
def get_company_financials(symbol: str) -> str:
    """Get company financial metrics: revenue, margins, growth and leverage.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Financial Metrics for {symbol}:\n"
            f"Revenue: {_num(info.get('totalRevenue'), money=True)}\n"
            f"Revenue Growth: {_num(info.get('revenueGrowth'), pct=True)}\n"
            f"Gross Profit: {_num(info.get('grossProfits'), money=True)}\n"
            f"Profit Margin: {_num(info.get('profitMargins'), pct=True)}\n"
            f"Operating Margin: {_num(info.get('operatingMargins'), pct=True)}\n"
            f"EBITDA: {_num(info.get('ebitda'), money=True)}\n"
            f"Earnings Growth: {_num(info.get('earningsGrowth'), pct=True)}\n"
            f"Return on Equity: {_num(info.get('returnOnEquity'), pct=True)}\n"
            f"Debt to Equity: {_num(info.get('debtToEquity'))}\n"
            f"Free Cash Flow: {_num(info.get('freeCashflow'), money=True)}"
        )
    except Exception as e:
        return f"Error fetching financial data for {symbol}: {e}"


print(get_stock_history(symbol="AAPL"))  # tools are still plain functions you can call

Historical Data for AAPL (1mo):
Starting Price: $341.01
Ending Price: $354.88
Change: +4.07%
Highest: $359.37
Lowest: $325.08
Average Volume: 47,513,013

Recent 5-day trend:
                                Close    Volume
Date                                           
2026-11-06 00:00:00-05:00  342.110046  44703800
2026-11-09 00:00:00-05:00  350.423737  48753100
2026-11-10 00:00:00-05:00  348.245392  29756100
2026-11-11 00:00:00-05:00  355.359985  34453500
2026-11-12 00:00:00-05:00  354.880005  31796600


In [16]:
financial_tools = [get_stock_price, get_stock_history, get_company_financials, web_search]

In [17]:
news_agent = Agent(
    name="news_agent",
    description="Researches recent news and developments about companies and industries.",
    model=model,
    system_prompt="""You are a news research specialist. Search for recent news and developments about companies and industries.

When searching:
- Focus on recent news (use time_range='week' or 'month')
- Prefer reliable sources such as major news outlets
- Summarize key developments and trends
- Include specific facts, dates, and figures when available
- Make at most two web_search calls: Groq's free tier caps each request at 8,000 tokens""",
    tools=[web_search],
    callback_handler=None,
)

financial_agent = Agent(
    name="financial_agent",
    description="Analyzes stock prices, price history, and company financial metrics.",
    model=model,
    system_prompt="""You are a financial analysis specialist. Analyze financial metrics, stock performance, and company valuations.

When analyzing:
- Use get_stock_price for current prices and key metrics
- Use get_stock_history for price trends
- Use get_company_financials for revenue, earnings, and growth
- Compare companies within the same sector and give specific numbers
- Look at no more than three companies, and use web_search at most once""",
    tools=financial_tools,
    callback_handler=None,
)

sentiment_agent = Agent(
    name="sentiment_agent",
    description="Analyzes market and investor sentiment from forums, social media, and commentary.",
    model=model,
    system_prompt="""You are a market sentiment analyst. Analyze market sentiment from social media, forums, and investor discussions.

When analyzing sentiment:
- Look for both bullish and bearish opinions
- Identify common concerns and excitement points
- Distinguish retail from institutional sentiment
- Summarize the overall market mood
- Make at most two web_search calls: Groq's free tier caps each request at 8,000 tokens""",
    tools=[web_search],
    callback_handler=None,
)
print("✓ Worker agents created")

✓ Worker agents created


In [18]:
supervisor = Agent(
    name="research_supervisor",
    model=model,
    system_prompt="""You are a market research coordinator. When asked about investment opportunities:

1. Work out what information is needed
2. Delegate to the specialist agents:
   - news_agent: recent developments and industry news
   - financial_agent: financial metrics and performance data
   - sentiment_agent: market sentiment and investor opinions
3. Synthesize their responses into one balanced recommendation

This is educational analysis, not financial advice — say so briefly at the end.
Do not ask the user clarifying questions: pick sensible defaults (for example, the sector's leading companies) and delegate straight away.""",
    tools=[news_agent.as_tool(), financial_agent.as_tool(), sentiment_agent.as_tool()],
)
print("✓ Supervisor created")

✓ Supervisor created


In [19]:
question = "Should I invest in electric vehicle companies?"
print(f"Question: {question}\n")

response = supervisor(question)

Question: Should I invest in electric vehicle companies?


Tool #1: news_agent

Tool #2: financial_agent

Tool #3: sentiment_agent


DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV stocks market investor sentiment Wall Street consensus hybrid demand 2024 2025                        │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] "Miles of Smiles Ahead” Stock Market (and Sentiment Results)… – Hedge Fund Tips                             │
│ URL: https://www.hedgefundtips.com/miles-of-smiles-ahead-stock-market-and-sentiment-results                     │
│ Score: 0.46657214                                                                                               │
│ Content: The hybrid pivot now underway in the U.S. is a best-case scenario for CPS, with hybrids accounting for │
│ a record 17.7% of Q3 retail sales and 15.7% of YTD sales (+3.1 ppts Y/Y). Demand continues to outrun supply,    │
│ with new hybrids selling in a median of 20 days versus 52 days for gas vehicles. Industry forecasts call for    │
│ hybrids to finish at ~17% of U.S. sales in 2026 and reach 32% by 2035, more than double the mix embedded in the │
│ 2030 plan. The number of hybrid models on the market is also [...] Hybrids are the fastest-growing powertrain   │
│ in the U.S.: 15.7% of sales YTD, up 3.1 points of share Y/Y. Q3 hybrid retail sales 575,246 (+28.0% Y/Y), a     │
│ record 17.7% of retail sales.                                                                                   │
│  Hybrids sell in a median 20 days vs 52 days for gas vehicles (Q3 2026); demand outrunning supply.              │
│  Hybrids forecast to grow from 14% of the U.S. market last year to ~34% by the end of the decade, absent        │
│ regulatory change [...] 2030 plan holds powertrain mix at 2025 levels, when hybrids (HEV + PHEV) were 14.3% of  │
│ U.S. sales. Forecast: 17% in 2026, 32% by 2035. Hybrid models available expected to roughly triple, from 37 in  │
│ 2025 to 100 in 2030 and 115 in 2033.                                                                            │
│  Shift already underway: Toyota RAV4 (#1 U.S. SUV) going hybrid-only in 2026, Jeep Cherokee returning as        │
│ hybrid-only in 2026 (Stellantis is ~10% of total revenue). 74% of CPS’s 2025 new awards were tied to battery    │
│ electric or full-hybrid platforms.                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Are Wall Street Analysts Bullish on Tesla Stock?                                                            │
│ URL: https://www.barchart.com/story/news/8552597/are-wall-street-analysts-bullish-on-tesla-stock                │
│ Score: 0.36375988                                                                                               │
│ Content: Tesla has struggled to win over investors amid intensifying competition in the EV market, with shares  │
│ tumbling12.6% over the past 52 weeks, sharply underperforming the S&P 500 Index ($SPX), which has gained 12%    │
│ over the same time frame. Over the past six months, however, both the stock and the index have rallied 8.2%.    │
│ [...] www.barchart.com                                                                                          │
│                                                                                                                 │
│ The ratings configuration is more bullish than it was 

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV electric vehicle tariffs subsidies price war Tesla BYD 2025                                           │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] 188K views · 1.7K reactions | BYD is going through concerning problems since 2024 because of which China    │
│ has changed its EV policy...                                                                                    │
│                                                                                                                 │
│ So the downfall came in three steps, and has created a new problem...                                           │
│                                                                                                                 │
│ Let’s understand the steps first.                                                                               │
│                                                                                                                 │
│ 1/ Supply Chain Financing: BYD took an average of 275 days to pay its suppliers in 2023... whereas most global  │
│ carmakers usually pay in under 90.                                                                              │
│                                                                                                                 │
│ In effect, suppliers were funding BYD’s growth... and in a way lending it money interest free to keep           │
│ operations going...                                                                                             │
│                                                                                                                 │
│ One research firm, GMT, argued that BYD’s true net debt was closer to 323 billion yuan, not the 27.7 billion    │
│ yuan it reported as of mid-2024...                                                                              │
│                                                                                                                 │
│ 2/ Chinese Subsidies: Apart from financing from suppliers... BYD received at least $3.7 billion in direct       │
│ subsidies, with aid jumping from €220 million in 2020 to €2.1 billion just two years later...                   │
│                                                                                                                 │
│ On top of that, China exempted EVs from purchase tax for years, which made every BYD cheaper for buyers as it   │
│ was a strategy for the country to flood the market with EVs...                                                  │
│                                                                                                                 │
│ 3/ With those two big sources of financing, came the third thing, which literally broke it... Zero-mileage      │
│ Sales...                                                                                                        │
│                                                                                                                 │
│ A number of Chinese companies were selling EVs to corporations in the thousands... and these new EVs would then │
│ just sit on parking lots collecting dust, not running... And some time later, the companies would put them on   │
│ the market for sale as second hand cars with no mileage on them...                                              │
│                                                       

DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: Tesla Ford GM EV strategy tariffs subsidies IRA 2025                                                     │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] In-Depth Analysis of Trump's Tariffs 2025 Policy | Gate Learn                                               │
│ URL: https://www.gate.com/learn/articles/in-depth-analysis-of-trump-s-tariffs-2025-policy/8186                  │
│ Score: 0.49757004                                                                                               │
│ Content: levels (such as Tesla and General Motors), while posing a heavy blow to brands heavily reliant on      │
│ imports (such as Hyundai and Toyota). Consumers may shift towards lower-priced used cars or domestic brands,    │
│ leading to a decline in imported car sales. The National Automobile Dealers Association (NADA) predicts an      │
│ overall sales decrease of 10%. [...] In the stock market, after the policy announcement, the three major US     │
│ stock indexes plummeted. On April 3rd, Trump announced tariffs, causing the Dow Jones Index to drop by 2.72%,   │
│ the S&P 500 to fall by 3.16%, and the Nasdaq to decline by 4.24%. Manufacturing companies such as General       │
│ Motors and Ford continued to be under pressure, and Tesla fell by over 7% due to its reliance on overseas parts │
│ supply chains. As a result, other major global stock markets also experienced declines. In the [...] Using      │
│ General Motors as an example, GM has a broad supply chain system globally, with some components imported from   │
│ countries such as China, Mexico, and Canada. After the tariff policy implementation, the cost of importing      │
│ components for GM has significantly increased. The tariff increase on some electronic components imported from  │
│ China has raised the cost of each component by around 25%. This not only increases production costs but also    │
│ affects the stability of the supply chain. Due to the                                                           │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Ford Vs GM EV Strategy: Who Can Build Cheaper EVs?                                                          │
│ URL: https://vroomhead.com/ford-vs-gm-ev-strategy                                                               │
│ Score: 0.4433293                                                                                                │
│ Content: The UEV platform is built around a simpler vehicle architecture and a new manufacturing process        │
│ intended to reduce production costs.                                                                            │
│                                                                                                                 │
│ GM is pursuing similar goals with its fourth-generation EV platform. However, its strategy relies heavily on    │
│ developing a new battery chemistry, making vehicles lighter and producing more efficient electric drive units.  │
│ The company has not yet revealed the full lineup that will use the new architecture.                            │
│                                                                                                                 │
│ ## Both Companies Are Adopting Tesla-Style Technology 

DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV sentiment retail investor forums charging hurdles hybrid demand Wall Street                           │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] OICA “5 News” —Wednesday, November 11, 2026 - International Organization of Motor Vehicle Manufacturers     │
│ URL: https://oica.net/oica-5-news-wednesday-november-11-2026                                                    │
│ Score: 0.44055954                                                                                               │
│ Content: U.S. EV market faces long slog as automakers shift to gas engines, hybrids after Trump rolls back      │
│ policies                                                                                                        │
│  U.S. EV registrations fell 31% in July as automakers shifted investment toward gasoline and hybrids. EV market │
│ share dropped to 6.2%, versus 16.4% for hybrids, although comparisons reflect last year’s buying rush before    │
│ tax credits expired. [...] Members only                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│ US auto sales September: hybrids, light trucks propel Toyota, Honda, Hyundai, Kia to September sales gains; GM  │
│ slips in Q3; Stellantis flat                                                                                    │
│  U.S. light-vehicle sales rose only 1.1% in Q3, with hybrids, affordable cars and crossovers driving gains for  │
│ Toyota, Honda, Hyundai and Kia. GM declined, Jeep weakened, and high prices pushed retail demand toward         │
│ smaller, efficient vehicles while fleets supported                                                              │
│  volume. [...] Lithium Plunges 25% in China on Fears Over Battery Demand Growth                                 │
│  Chinese lithium carbonate futures fell about 25% in September as investors questioned whether EV and           │
│ energy-storage demand can absorb expanding battery output. Excess capacity, inventories and weaker EV growth    │
│ are weighing on sentiment, though some analysts still expect near-term supply deficits.                         │
│                                                                                                                 │
│                                                                                                                 │
│ Read more                                                                                                       │
│                                                                                                                 │
│ ### Francois Roudier                                                                                            │
│                                                                                                                 │
│ September 29, 2026                                                                                              │
│                                                                                                                 │
│                                                       

╭───────────────────────────────────────────── Tavily Search Results ─────────────────────────────────────────────╮
│ Query: EV sector financial health margins price war 2025 2026                                                   │
│                                                                                                                 │
│ Results: 3 found                                                                                                │
│ --------------------------------------------------                                                              │
│                                                                                                                 │
│ [1] BYD Earnings Miss Estimates as China's EV Price War Takes Its Toll                                          │
│ URL: https://finance.yahoo.com/markets/stocks/articles/byd-earnings-miss-estimates-chinas-190000731.html        │
│ Score: 0.5173271                                                                                                │
│ Content: Despite the quarterly improvement, BYD's net income for the first nine months of 2026 declined 4.2% to │
│ 22.34 billion yuan, while revenue remained virtually unchanged at 566.3 billion yuan.                           │
│                                                                                                                 │
│ However, BYD isn't the only Chinese automaker feeling the pressure.                                             │
│                                                                                                                 │
│ Domestic rival Geely Automobile also reported third-quarter results on Thursday, with net profit climbing 28%   │
│ to 4.84 billion yuan ($726 million), below analysts' expectations of 5.19 billion yuan. [...] The European      │
│ Union has already imposed additional tariffs on Chinese-made battery-electric vehicles and is considering       │
│ measures to restrict rapidly growing Chinese hybrid imports.                                                    │
│                                                                                                                 │
│ BYD is responding by expanding its manufacturing footprint in Europe, with its Hungarian factory expected to    │
│ begin mass production before the end of 2026.                                                                   │
│                                                                                                                 │
│ Geely, in turn, is also expanding internationally, including through a manufacturing partnership with Ford in   │
│ Spain. [...] BYD's net profit rose 28% year-over-year to 10.02 billion yuan ($1.5 billion), missing analysts'   │
│ expectations of approximately 11.1 billion yuan. Revenue increased 13.6% to 221.5 billion yuan ($33.3 billion), │
│ also falling short of market forecasts.                                                                         │
│                                                                                                                 │
│ The earnings miss comes as the Chinese automaker struggles with fierce competition and slowing demand in its    │
│ domestic market, increasingly relying on higher-margin international sales to support profitability.            │
│                                                                                                                 │
│                                                                                                                 │
│ [2] Electric Vehicle Market Size And Share Report, 2026-2033                                                    │
│ URL: https://www.grandviewresearch.com/industry-analysis/electric-vehicles-ev-market                            │
│ Score: 0.38867602                                                                                               │
│ Content: to own an EV. This high disparity in the init

### Executive Summary: The EV Market in Transition

The global electric vehicle (EV) sector has moved from early-stage speculative euphoria (2020–2022) into a **mature, hyper-competitive cyclical shakeout**. 

While long-term secular adoption remains intact, the industry is currently navigating an **"adoption chasm"**: early tech adopters have largely bought in, while the mass-market consumer remains hesitant due to higher upfront vehicle prices, charging infrastructure deficits, elevated insurance premiums, and steep depreciation. As a result, investing in EV companies today requires a selective, risk-aware approach rather than a broad-brush sector bet.

---

### 1. Industry Dynamics & Recent Developments

* **Intense Price Wars & Margin Squeeze:** Initiated by aggressive discounting in China (led by BYD) and globally by Tesla, vehicle price reductions of 10%–20% have become common. While driving market penetration, this price deflation has severely compressed operating and gross marg

In [ ]:
questions = [
    "What's the outlook for renewable energy stocks?",
    "Is the semiconductor industry a good investment right now?",  # see the Groq note above
]

for q in questions:
    print(f"\n{'=' * 80}\nQuestion: {q}\n{'=' * 80}")
    supervisor.messages = []  # start each question fresh (see the Groq note above)
    supervisor(q)